# Fermion-to-qubit encoding: FermionicSum + Jordan-Wigner

Walkthrough of `core/fermions.py` (Task 20). The point of this module is that a fermionic model is defined once, encoding-agnostic, and only becomes qubits when you explicitly ask via `.encode(...)`. `FermionicHamiltonian` -- a `FermionicSum` that asserts Hermiticity and can carry a fermionic-native `trotter_groups` -- shows up partway through, wherever the example needs a genuine ground state or Trotter circuit.

In [1]:
import warnings 
warnings.filterwarnings("ignore")

In [2]:
import numpy as np

from qaravan.applications import ground_state
from qaravan.core.fermions import (
    FermionicSum,
    FermionicEncoding,
    hopping,
    interleaved_mode_order,
    majorana_bilinear,
)
from qaravan.core.hamiltonians import FermionicHamiltonian

from qaravan.plot import set_style

np.set_printoptions(precision=4, suppress=True)

## 1. A bare fermionic Hamiltonian: just hopping terms

`FermionicSum` is pure algebra -- a sum of `FermionTerm`s (products of ladder operators), with no notion of qubits at all, and no Hermiticity claim (it can hold a single bare, non-Hermitian ladder operator). `FermionicHamiltonian(FermionicSum)` adds exactly one thing: an assertion, checked at construction, that the terms *are* Hermitian -- what a genuine physical Hamiltonian needs to be, and what lets `.encode()` hand back a full `Hamiltonian` (with `ground_state()`/`trotter_circuit()` available) instead of just a `PauliSum`. Since we want this chain's ground state below, we build it as a `FermionicHamiltonian`.

`hopping(a, b, coeff)` returns the two `FermionTerm`s making up `coeff * (c†_a c_b + c†_b c_a)`.

Here's a 4-mode chain with nearest-neighbor hopping only (no interaction, no chemical potential) -- deliberately simpler than `FermiHubbard1D`, to see the bare mechanism before the model-specific machinery.

In [3]:
num_modes = 4
t = 1.0

hopping_terms = []
for site in range(num_modes - 1):
    hopping_terms += hopping(site, site + 1, -t)
hopping_terms

[-1.0 * c†_0 c_1,
 -1.0 * c†_1 c_0,
 -1.0 * c†_1 c_2,
 -1.0 * c†_2 c_1,
 -1.0 * c†_2 c_3,
 -1.0 * c†_3 c_2]

In [4]:
fermion_ham = FermionicHamiltonian(hopping_terms, num_modes)
fermion_ham

-1.0 * c†_0 c_1 + -1.0 * c†_1 c_0 + -1.0 * c†_1 c_2 + -1.0 * c†_2 c_1 + -1.0 * c†_2 c_3 + -1.0 * c†_3 c_2

## 2. An encoding: Jordan-Wigner, interleaved order

`FermionicEncoding` is a value -- which map (`"jordan_wigner"`, for now) and which qubit each mode lands on (`mode_order[qubit] = mode`). It carries no reference to any particular Hamiltonian; the same encoding can be reused for any operator over the same `num_modes`.

"Interleaved" here just means a non-trivial permutation -- unlike `FermiHubbard1D`'s own `interleaved_mode_order` (which encodes a specific spin-pairing convention), this is a generic example: swap each adjacent pair of modes.

In [5]:
# qubit 0 <- mode 1, qubit 1 <- mode 0, qubit 2 <- mode 3, qubit 3 <- mode 2
# note that the codebase assumes a default ordering for the modes: c1up, c2up, c3up, ..., c1down, c2down, c3down, ...
# and then deciding to map that MODE ordering to a qubit ordering 

interleaved_order = tuple(m for pair in zip(range(1, num_modes, 2), range(0, num_modes, 2)) for m in pair)
encoding = FermionicEncoding("jordan_wigner", num_modes=num_modes, mode_order=interleaved_order)
encoding

FermionicEncoding(name='jordan_wigner', num_modes=4, mode_order=(1, 0, 3, 2))

## 3. Apply the encoding: the result is just Paulis

`.encode()` is the *only* qubit-producing method in this whole module. Everything above (`FermionTerm`, `FermionicSum`) never mentioned a qubit; the object below is a plain `Hamiltonian` -- indistinguishable from one you'd get by hand-typing a `PauliSum`.

In [6]:
qubit_ham = fermion_ham.encode(encoding)
print(type(qubit_ham).__name__)  # a plain Hamiltonian -- no fermionic residue
qubit_ham

Hamiltonian


((-0.5+0j))*XXII + ((-0.5+0j))*YYII + ((-0.5+0j))*YZZY + ((-0.5+0j))*XZZX + ((-0.5+0j))*IIXX + ((-0.5+0j))*IIYY

### Contrast: a bare `FermionicSum`'s `.encode()` returns a `PauliSum`, not a `Hamiltonian`

`fermion_op` above is a `FermionicHamiltonian`, so `.encode()` returned a full `Hamiltonian`. A plain `FermionicSum` has no `trotter_groups` concept at all -- it's pure algebra, with no claim about being a physical Hamiltonian -- so its `.encode()` hands back just the `PauliSum` representation directly, with nothing extra tacked on.

In [7]:
bare_op = FermionicSum(hopping(0, 1, -1.0), num_fermionic_modes=2)  # note: FermionicSum, not FermionicHamiltonian
bare_encoded = bare_op.encode(encoding_2q := FermionicEncoding("jordan_wigner", num_modes=2))
print(type(bare_encoded).__name__)
bare_encoded

PauliSum


((-0.5+0j))*YY + ((-0.5+0j))*XX

## 4. `.as_matrix()` gives a matrix on the correct number of qubits

In [8]:
H = qubit_ham.as_matrix()
print("shape:", H.shape, "expected:", (2**encoding.num_qubits, 2**encoding.num_qubits))
assert H.shape == (2**encoding.num_qubits, 2**encoding.num_qubits)
assert np.allclose(H, H.conj().T)  # Hermitian, as guaranteed by jordan_wigner's pairing
print("Hermitian: OK")

shape: (16, 16) expected: (16, 16)
Hermitian: OK


## 5. `ground_state()` gives a `Statevector`

Ordinary exact diagonalization -- `applications.ground_state()` doesn't know or care that this particular `Hamiltonian` came from encoding a `FermionicSum`. It lives in `applications/`, not as a `Hamiltonian` method, because it has to hand back a backend `Statevector` -- `core/` doesn't depend on `backends/`.

In [9]:
gs = ground_state(qubit_ham)
print(type(gs).__name__)
gs

Statevector


0.2236|0011⟩ + -0.5000|0101⟩ + -0.4472|0110⟩ + -0.4472|1001⟩ + -0.5000|1010⟩ + 0.2236|1100⟩

## 6. Create a Majorana bilinear as an observable

Same split as before: define the *physics* of the observable (`i·γ_a·γ_b`, a `MajoranaTerm`) with no reference to qubits yet.

In [10]:
majorana_obs = [majorana_bilinear(0, 3)]  # i * gamma_0 * gamma_1, on mode 0
majorana_obs

[1j * γ_0 γ_3]

## 7. Apply the *same* encoding to the Majorana observable -- a separate step

This is the point of `FermionicEncoding`: the exact same `encoding` object used for the Hamiltonian above is reused here, so the observable and the ground state it's about to be evaluated on are guaranteed to agree on which qubit is which mode.

In [11]:
encoded_majorana_obs = encoding.encode_majorana_terms(majorana_obs)
print(type(encoded_majorana_obs).__name__)
encoded_majorana_obs

PauliSum


XXII

## 8. Expectation value of the encoded Majorana operator in the ground state

`encoded_majorana_obs` is a plain `PauliSum` (an `Observable`), so `Statevector.expectation` handles it exactly as it would any hand-written Pauli observable -- no new backend code anywhere.

In [12]:
value = gs.expectation(encoded_majorana_obs)
print(f"<i*gamma_0*gamma_1> in the ground state = {value:.6f}")
assert abs(value.imag) < 1e-10  # Hermitian observable -> real expectation value

<i*gamma_0*gamma_1> in the ground state = 0.894427+0.000000j


## 9. A spinless dimer: single fermion -> symmetric superposition

The simplest possible check that all of this reproduces textbook physics: two modes, pure hopping, no interaction. In the one-particle sector the Hamiltonian is just the 2x2 hopping matrix `[[0, -t], [-t, 0]]`, whose ground state (for `t > 0`) is the *symmetric* (bonding) combination of "fermion on site 0" and "fermion on site 1", with energy `-t`.

In [13]:
t = 1.0
dimer = FermionicHamiltonian(hopping(0, 1, -t), num_fermionic_modes=2)
dimer

-1.0 * c†_0 c_1 + -1.0 * c†_1 c_0

In [14]:
dimer_encoding = FermionicEncoding("jordan_wigner", num_modes=2)  # default (identity) mode_order
dimer_encoding
dimer_ham = dimer.encode(dimer_encoding)
dimer_ham.as_observable()

((-0.5+0j))*YY + ((-0.5+0j))*XX

In [15]:
E0 = np.linalg.eigvalsh(dimer_ham.as_matrix())[0]
print(f"ground energy = {E0:.6f}  (expected -t = {-t:.6f})")
assert np.isclose(E0, -t)

ground energy = -1.000000  (expected -t = -1.000000)


In [16]:
dimer_gs = ground_state(dimer_ham, fixed_particle_number=1)
dimer_gs # uniform superposition over first site being occupied and second site being occupied

-0.7071|01⟩ + -0.7071|10⟩

## 10. Spin-labelled modes: which qubit carries which spin

Everything above is spin-agnostic -- a mode is just an index. Spinful models in qaravan share one
labelling convention: **modes $[0, M)$ are up, $[M, 2M)$ are down**, which is what
`FermiHubbard1D` ($\sigma L + j$) and `AndersonImpurity` ($\sigma(N_\text{bath}+1) + \text{site}$)
both use when they place their terms.

That convention lives in the *model*. The mode $\to$ qubit permutation lives in the *encoding*.
`spin_qubits()` composes the two, so you never hand-write a qubit list:

In [17]:
from qaravan.core.fermions import interleaved_mode_order

for label, mode_order in [("identity", None), ("interleaved", interleaved_mode_order(3))]:
    spinful_encoding = FermionicEncoding("jordan_wigner", num_modes=6, mode_order=mode_order)
    up, down = spinful_encoding.spin_qubits()
    print(f"{label:12s} up qubits {up}   down qubits {down}")

identity     up qubits [0, 1, 2]   down qubits [3, 4, 5]
interleaved  up qubits [0, 2, 4]   down qubits [1, 3, 5]


`mode_order` is a permutation of the *canonical* labelling, so the up/down split in mode space is
unchanged by it -- only the qubits those modes land on move. Note this is not a mere relabelling
of the qubit register: `_majorana_pauli_string` builds the JW parity string over **qubit** indices,
so a permuted `mode_order` is a genuinely different (isospectral) encoding with different Pauli
weights.

It raises on an odd `num_modes`, but an even-mode *spinless* operator (the 4-mode chain from §1,
say) will answer happily -- only ask when spin labelling actually applies.

## Summary

- `FermionicSum`/`MajoranaSum` are pure algebra: no qubits, no encoding, until `.encode(...)` is called.
- `FermionicEncoding` is a value (map name + `mode_order`) built once and reused for both a Hamiltonian and any observable evaluated on its ground state -- the two can't silently drift onto different encodings.
- `jordan_wigner` guarantees every emitted `HamiltonianTerm` is Hermitian by auto-pairing non-self-adjoint terms (e.g. the two halves of a hopping bond); it raises `ValueError` if a term has no partner, rather than silently handing `trotter_circuit()` a non-unitary gate.
- `FermionicHamiltonian(FermionicSum)` (used by `FermiHubbard1D`/`AndersonImpurity`) asserts Hermiticity at construction, and carries an optional `trotter_groups` -- a fermionic-native, pre-encoding analog of `Hamiltonian.trotter_groups`. `.encode()` maps each group through Jordan-Wigner independently (not the flattened term list), so the grouping survives into the encoded `Hamiltonian` for free, with no post-encoding re-chunking.
- Because of that split, `.encode()` returns *different* types depending on which class you called it on: `FermionicSum.encode() -> PauliSum` (no execution-strategy concept to carry), `FermionicHamiltonian.encode() -> Hamiltonian` (carries `trotter_groups`, enables `ground_state()`/`trotter_circuit()`). A bare `FermionicSum` was never claiming to be a Hamiltonian, so its `.encode()` shouldn't hand one back either.